# 2.4 Résolution par Colonies de Fourmis (Ant Colony Optimization)

Ce notebook implémente un troisième algorithme pour le CVRP : **Ant Colony Optimization (ACO)**.

## Principe

ACO est une métaheuristique inspirée du comportement des fourmis réelles : elles déposent des phéromones sur leurs chemins, et les pistes courtes voient leurs phéromones s'accumuler plus vite, créant une convergence collective vers les meilleurs trajets.

Pour le CVRP, chaque *fourmi artificielle* construit une solution complète (un ensemble de tournées) de la manière suivante :

1. Partir du dépôt avec un véhicule vide.
2. Choisir la prochaine ville non visitée selon la règle probabiliste :
$$P(i \to j) = \frac{[\tau_{ij}]^{\alpha} \cdot [\eta_{ij}]^{\beta}}{\sum_{l \in \text{faisable}} [\tau_{il}]^{\alpha} \cdot [\eta_{il}]^{\beta}}$$
   où $\tau_{ij}$ est le niveau de phéromone et $\eta_{ij} = 1/c_{ij}$ est l'information heuristique (inverse de la distance).
3. Si aucune ville ne rentre dans la capacité restante, retour au dépôt et ouverture d'un nouveau véhicule.
4. Continuer jusqu'à avoir visité toutes les villes.

À chaque itération, les phéromones s'évaporent ($\tau \leftarrow (1-\rho)\tau$) puis les fourmis déposent une quantité proportionnelle à la qualité de leur solution ($\Delta\tau = Q/L$, où $L$ est la distance totale).

## Paramètres
- $\alpha$ : poids des phéromones (exploitation du savoir accumulé)
- $\beta$ : poids de l'heuristique (attirance des villes proches)
- $\rho$ : taux d'évaporation (entre 0 et 1)
- $Q$ : constante de dépôt
- `n_fourmis` : nombre de fourmis par itération
- `n_iter` : nombre d'itérations

## Préparation — imports et génération d'instance

Reprise des mêmes conventions que `ProjetFinal.ipynb` : solution = liste de tournées, chaque tournée commence et finit au dépôt (sommet 0).

In [ ]:
import math
import random
import time
import statistics
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

In [ ]:
def generer_instance(n=None, nb_vehicules=None, n_max=15, nb_vehicules_max=10, max_capacite_coeff=2.5):
    n = random.randint(5, n_max) if n is None else int(n)
    nb_vehicules = random.randint(2, nb_vehicules_max) if nb_vehicules is None else int(nb_vehicules)
    taille = n + 1
    matrice = [[0] * taille for _ in range(taille)]
    for i in range(taille):
        for j in range(i + 1, taille):
            dist = random.randint(5, 50) if random.random() > 0.1 else float('inf')
            matrice[i][j] = dist
            matrice[j][i] = dist
    demandes = [0]
    total_colis = 0
    for _ in range(n):
        d = random.randint(1, 10)
        demandes.append(d)
        total_colis += d
    capacite_min = math.ceil(total_colis / nb_vehicules)
    capacite_max = int(capacite_min * random.uniform(1.1, max_capacite_coeff))
    print(f"\n--- Instance générée ---")
    print(f"Total colis à livrer : {total_colis}")
    print(f"Capacité par véhicule : {capacite_max}")
    return n, nb_vehicules, matrice, demandes, capacite_max

def distance_total(solution, matrice):
    z = 0
    for tournee in solution:
        for i in range(len(tournee) - 1):
            z += matrice[tournee[i]][tournee[i+1]]
    return z

def verifier_solution(solution, matrice, demandes, capacite_max):
    n = len(matrice)
    villes_visitees = []
    for k, tournee in enumerate(solution):
        if len(tournee) <= 2:
            continue
        if tournee[0] != 0 or tournee[-1] != 0:
            return False
        charge = 0
        for i in range(len(tournee) - 1):
            u, v = tournee[i], tournee[i+1]
            if matrice[u][v] == float('inf'):
                return False
            if v != 0:
                villes_visitees.append(v)
                charge += demandes[v]
        if charge > capacite_max:
            return False
    return sorted(villes_visitees) == list(range(1, n))


## Implémentation d'ACO

### Construction d'une solution par une fourmi

Une fourmi construit une solution en enchaînant des tournées. Pour chaque ville à visiter, elle calcule la probabilité de chaque ville candidate (non visitée, rentrant dans la capacité restante, accessible) et tire au sort.

In [ ]:
def _construire_solution_fourmi(matrice, demandes, capacite_max, nb_vehicules, pheromones, alpha, beta):
    """Une fourmi construit une solution complète (liste de tournées)."""
    n = len(matrice) - 1  # nombre de villes (hors dépôt)
    non_visitees = set(range(1, n + 1))
    solution = []
    vehicules_utilises = 0

    while non_visitees and vehicules_utilises < nb_vehicules:
        tournee = [0]
        charge = 0
        courant = 0

        while True:
            # Villes candidates : non visitées, tenant dans la capacité, accessibles
            candidates = [
                j for j in non_visitees
                if charge + demandes[j] <= capacite_max and matrice[courant][j] != float('inf')
            ]
            if not candidates:
                break

            # Calcul des probabilités (phéromone^alpha * heuristique^beta)
            poids = []
            for j in candidates:
                tau = pheromones[courant][j] ** alpha
                eta = (1.0 / matrice[courant][j]) ** beta
                poids.append(tau * eta)

            total = sum(poids)
            if total == 0:
                choix = random.choice(candidates)
            else:
                r = random.random() * total
                cumul = 0
                choix = candidates[-1]
                for j, p in zip(candidates, poids):
                    cumul += p
                    if cumul >= r:
                        choix = j
                        break

            tournee.append(choix)
            charge += demandes[choix]
            non_visitees.remove(choix)
            courant = choix

        # Fermeture de la tournée (retour au dépôt si accessible)
        if matrice[courant][0] == float('inf'):
            return None  # solution infaisable
        tournee.append(0)
        solution.append(tournee)
        vehicules_utilises += 1

    if non_visitees:
        return None  # on a épuisé les véhicules avant de tout visiter

    # Complète avec des tournées vides pour atteindre nb_vehicules
    while len(solution) < nb_vehicules:
        solution.append([0, 0])

    return solution

### Boucle principale ACO

À chaque itération : toutes les fourmis construisent une solution, on évapore les phéromones, puis on dépose en fonction de la qualité (règle *Ant System* avec renforcement élitiste de la meilleure solution globale).

In [ ]:
def aco_cvrp(matrice, demandes, capacite_max, nb_vehicules,
             n_fourmis=20, n_iter=100,
             alpha=1.0, beta=3.0, rho=0.1, Q=100.0,
             tau_0=1.0, elitisme=True, historique=False):
    """
    Résout le CVRP par Ant Colony Optimization.
    Retourne (meilleure_solution, meilleure_distance) ou (sol, dist, hist_courante, hist_meilleure) si historique=True.
    """
    taille = len(matrice)
    pheromones = [[tau_0] * taille for _ in range(taille)]

    meilleure_solution = None
    meilleure_distance = float('inf')
    hist_courante = []
    hist_meilleure = []

    for it in range(n_iter):
        solutions_iter = []
        for _ in range(n_fourmis):
            sol = _construire_solution_fourmi(
                matrice, demandes, capacite_max, nb_vehicules,
                pheromones, alpha, beta
            )
            if sol is None:
                continue
            dist = distance_total(sol, matrice)
            solutions_iter.append((sol, dist))
            if dist < meilleure_distance:
                meilleure_distance = dist
                meilleure_solution = sol

        # Évaporation
        for i in range(taille):
            for j in range(taille):
                pheromones[i][j] *= (1 - rho)

        # Dépôt proportionnel à la qualité de chaque fourmi
        for sol, dist in solutions_iter:
            depot = Q / dist if dist > 0 else 0
            for tournee in sol:
                for k in range(len(tournee) - 1):
                    u, v = tournee[k], tournee[k+1]
                    pheromones[u][v] += depot
                    pheromones[v][u] += depot

        # Renforcement élitiste : boost supplémentaire sur la meilleure globale
        if elitisme and meilleure_solution is not None:
            depot_elite = Q / meilleure_distance
            for tournee in meilleure_solution:
                for k in range(len(tournee) - 1):
                    u, v = tournee[k], tournee[k+1]
                    pheromones[u][v] += depot_elite
                    pheromones[v][u] += depot_elite

        if historique:
            dist_iter = min((d for _, d in solutions_iter), default=float('inf'))
            hist_courante.append(dist_iter)
            hist_meilleure.append(meilleure_distance)

    if historique:
        return meilleure_solution, meilleure_distance, hist_courante, hist_meilleure
    return meilleure_solution, meilleure_distance

## Démonstration sur une instance aléatoire

In [ ]:
random.seed(42)
n, nb_v, mat, dems, cap = generer_instance(n=10, nb_vehicules=3)

t0 = time.time()
sol, dist, hist_c, hist_m = aco_cvrp(
    mat, dems, cap, nb_v,
    n_fourmis=20, n_iter=80,
    alpha=1.0, beta=3.0, rho=0.1, Q=100.0,
    historique=True,
)
duree = time.time() - t0

print(f"\nDistance ACO  : {dist}")
print(f"Solution      : {sol}")
print(f"Valide        : {verifier_solution(sol, mat, dems, cap)}")
print(f"Durée         : {duree:.3f} s")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hist_c, label="Meilleure de l'itération", alpha=0.6)
plt.plot(hist_m, label="Meilleure globale", linewidth=2)
plt.xlabel("Itération")
plt.ylabel("Distance totale")
plt.title("Convergence de l'ACO sur le CVRP")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show();

## Étude de sensibilité aux paramètres

Petite exploration de l'impact de $\beta$ (poids de l'heuristique) sur la qualité finale, à $\alpha$, $\rho$ et $Q$ fixés.

In [ ]:
betas = [0.5, 1.0, 2.0, 3.0, 5.0]
n_tests = 5
resultats = {b: [] for b in betas}

for b in betas:
    for t in range(n_tests):
        random.seed(100 + t)
        _, _, m, d, c = generer_instance(n=10, nb_vehicules=3)
        _, dist = aco_cvrp(m, d, c, 3, n_fourmis=15, n_iter=50, beta=b)
        resultats[b].append(dist)

moyennes = [statistics.mean(resultats[b]) for b in betas]
print("beta | distance moyenne")
for b, m in zip(betas, moyennes):
    print(f"{b:4} | {m:.1f}")

plt.figure(figsize=(8, 4))
plt.plot(betas, moyennes, marker='o')
plt.xlabel("beta (poids de l'heuristique)")
plt.ylabel("Distance moyenne (sur {} instances)".format(n_tests))
plt.title("Sensibilité d'ACO au paramètre beta")
plt.grid(True, alpha=0.3)
plt.show();